In [ ]:
import numpy as np
import matplotlib.pyplot as plt

STUDENT_ID = 2021030028

#### Explore then Exploit Algorithm

In [ ]:
class EtE:
    def __init__(self, n_arms, N):
        self.n_arms = n_arms
        self.N = N
        self.explore_steps = n_arms * N
        self.exploring = True
        self.values = np.zeros(n_arms)
        self.counts = np.zeros(n_arms)

    def select_arm(self):
        t = int(np.sum(self.counts))
        if t < self.explore_steps:
            return t % self.n_arms
        self.exploring = False
        return int(np.argmax(self.values))

    def update(self, chosen_arm, reward):
        if self.exploring:
            self.counts[chosen_arm] += 1
            n = self.counts[chosen_arm]
            self.values[chosen_arm] += (reward - self.values[chosen_arm]) / n

#### Upper Confidence Bound Algorithm

In [ ]:
class UCB:
    def __init__(self, n_arms, max_reward=1):
        self.counts = np.zeros(n_arms)
        self.values = np.zeros(n_arms)
        self.arms = n_arms
        self.max_reward = max_reward

    def select_arm(self):
        for arm in range(self.arms):
            if self.counts[arm] == 0:
                return arm
        t = np.sum(self.counts)
        ucb_values = self.values + self.max_reward * np.sqrt((2 * np.log(t)) / self.counts)
        return int(np.argmax(ucb_values))

    def update(self, chosen_arm, reward):
            self.counts[chosen_arm] += 1
            n = self.counts[chosen_arm]

            value = self.values[chosen_arm]
            self.values[chosen_arm] = value + ((reward - value) / n)
            return

#### Multiplicative Weights Update Algorithm
###### MWU is wrapped inside a wrapper class that converts the algorithm from Experts to Bandits setting
###### and from cost based to reward based weight updating

In [ ]:
class MWU:
    def __init__(self, experts, h, scale=1.0, seed=28):
        self.weights = np.ones(experts)
        self.h = h
        self.scale = scale
        self.prob_vector = self.weights / np.sum(self.weights)
        self.random = np.random.default_rng(seed)

    def select_expert(self):
        selected = self.random.choice(np.size(self.weights), p=self.prob_vector)
        return selected

    def update(self, cost_vector):
        self.weights *= np.power((1 - self.h), (cost_vector / self.scale))
        self.prob_vector = self.weights / np.sum(self.weights)


class MWU_bandit:
    def __init__(self, n_arms, h, gamma, seed=28):
        self.mwu = MWU(n_arms, h, scale=(n_arms/gamma), seed=seed)
        self.arms = n_arms
        self.gamma = gamma
        self.curr_q = 1.0

    def select_arm(self):
        q = ((1 - self.gamma) * self.mwu.prob_vector) + (self.gamma/self.arms)
        selected = np.random.choice(np.size(q), p=q)
        self.curr_q = q[selected]
        return selected

    def update(self, chosen_arm, reward):
        cost_vector = np.zeros(self.arms)
        cost = -reward
        cost_vector[chosen_arm] = cost / self.curr_q
        self.mwu.update(cost_vector)

    def get_probabilities(self):
        return ((1 - self.gamma) * self.mwu.prob_vector) + (self.gamma / self.arms)

# Part A

#### Environment implementation

###### Environment class

In [ ]:
class BanditEnvironment:
    def __init__(self, p_L, p_H, T_L, T_H, initial_states):
        self.p_L = np.array(p_L)
        self.p_H = np.array(p_H)
        self.T_L = np.array(T_L)
        self.T_H = np.array(T_H)
        self.initial_states = np.array(initial_states)
        self.k_arms = len(p_L)
        self.t_curr = 0

    def _get_curr_prob(self, arm):
        period = self.T_L[arm] + self.T_H[arm]

        time_in_period = self.t_curr % period
        starts_low = (self.initial_states[arm] == 'L')

        if starts_low:
            is_low = time_in_period < self.T_L[arm]
        else:
            is_low = time_in_period >= self.T_H[arm]

        return self.p_L[arm] if is_low else self.p_H[arm]

    def pull(self, arm):
        p = self._get_curr_prob(arm)

        reward = np.random.binomial(1, p)
        self.t_curr += 1

        return reward

###### Environment simulation

In [ ]:
def mc_simulation(algorithms, env_params, T, epochs):
    results = {
        name: np.zeros((epochs, T)) for name in algorithms.keys()
    }

    selections = {
        name: np.zeros((epochs, T)) for name in algorithms.keys()
    }
    
    for epoch in range(epochs):
        for name, algorithm in algorithms.items():
            
            np.random.seed(STUDENT_ID + epoch) # Call random.seed inside each algorithm to ensure same sequence
            
            env = BanditEnvironment(**env_params)

            algo_class = algorithm['class']
            algo_params = algorithm['params']
            algo = algo_class(**algo_params)

            cum_reward = 0

            for t in range(T):
                
                arm = algo.select_arm()
                reward = env.pull(arm)
                algo.update(arm, reward)

                cum_reward += reward
                results[name][epoch, t] = cum_reward
                selections[name][epoch, t] = arm

    return results, selections

###### Figure plotting

In [ ]:
def plot_cumulative_rewards(sim_results, title="Cumulative Rewards", filename=None):
    plt.figure(figsize=(10, 6))

    for i, (name, data) in enumerate(sim_results.items()):
        mean_curve = np.mean(data, axis=0)
        final_values = data[:, -1]
        final_mean = np.mean(final_values)
        final_std = np.std(final_values)
        
        print(f"{name} Final Reward: {final_mean:.2f} ± {final_std:.2f}")
        
        # Plot all curves on the same figure
        plt.plot(mean_curve, label=name, color=f'C{i}')

    plt.xlabel('Time Step (t)')
    plt.ylabel('Cumulative Reward')
    plt.title(title)
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.7) 
    plt.tight_layout()

    # Save only if a filename is provided
    if filename:
        plt.savefig(filename, format="pdf", bbox_inches="tight")
        print(f"Figure saved as: {filename}")

    plt.show()

def plot_arm_selections(arm_selections, max_time=None, num_bins=20, title="Arm Selections", filename=None):
    num_algos = len(arm_selections)
    fig, axes = plt.subplots(num_algos, 1, figsize=(10, 4 * num_algos), sharex=True)
    if num_algos == 1: axes = [axes]

    for i, (name, data) in enumerate(arm_selections.items()):
        ax = axes[i]
        
        # Slice data if a time limit is set
        current_data = data[:, :max_time] if max_time else data
            
        _, time_idx_arm0 = np.where(current_data == 0)
        _, time_idx_arm1 = np.where(current_data == 1)
        
        ax.hist(
            [time_idx_arm0, time_idx_arm1], 
            bins=num_bins, 
            color=['C0', 'C1'], 
            label=['Arm 0', 'Arm 1'], 
            edgecolor='black', 
            alpha=0.7, 
            stacked=True
        )
        
        ax.set_title(f'{name} - {title}')
        ax.set_ylabel('Total Pulls')
        ax.legend(loc='upper right')

    axes[-1].set_xlabel('Time Step (t)')
    fig.suptitle(title)
    plt.tight_layout()

    # Save only if a filename is provided
    if filename:
        plt.savefig(filename, format="pdf", bbox_inches="tight")
        print(f"Figure saved as: {filename}")

    plt.show()

#### A.1 Sanity check

In [ ]:
T = 50000
epochs = 50

n_arms = 2
sanity_env_params = {
    'p_L': [0.4, 0.8],
    'p_H': [0.4, 0.8],
    'T_L': [100, 100],
    'T_H': [100, 100],
    'initial_states': ['L', 'L']
}

algorithms = {
    'Explore then Exploit': {
        'class': EtE,
        'params': {'n_arms': n_arms, 'N': np.power((T/n_arms), (2/3))}
    },
    'Upper Confidence Bound': {
        'class': UCB,
        'params': {'n_arms': n_arms}
    },
    'Multiplicative Weights in Bandit mode': {
        'class': MWU_bandit,
        'params': {'n_arms': n_arms, 'h': np.power(((n_arms * np.log(n_arms)) / T) , (1/3)), 'gamma': np.power(((n_arms * np.log(n_arms)) / T) , (1/3))}
    }
}

sim_results, arm_selections = mc_simulation(algorithms, sanity_env_params, T, epochs)

In [ ]:
plot_cumulative_rewards(sim_results, title='Task A.1: Cumulative Rewards', filename='cum_reward_a1.pdf')
plot_arm_selections(arm_selections, max_time=None, num_bins=200, title='Task A.1: General Histogram', filename='selections_a1.pdf')
plot_arm_selections(arm_selections, max_time=2000, num_bins=200, title='Task A.1: Early Phase Histogram', filename='early_a1.pdf')

#### A.2 Instance where MW beats Explore-then-Exploit

In [ ]:
T = 50000
epochs = 50

n_arms = 2

explore_steps = int((np.power((T/n_arms), (2/3))) * n_arms + 1)

sanity_env_params = {
    'p_L': [0.1, 0.1],
    'p_H': [0.9, 0.9],
    'T_L': [(T - explore_steps), explore_steps],
    'T_H': [explore_steps, (T - explore_steps)],
    'initial_states': ['H', 'L']
}

algorithms = {
    'Explore then Exploit': {
        'class': EtE,
        'params': {'n_arms': n_arms, 'N': np.power((T/n_arms), (2/3))}
    },
    'Multiplicative Weights in Bandit mode': {
        'class': MWU_bandit,
        'params': {'n_arms': n_arms, 'h': np.power(((n_arms * np.log(n_arms)) / T) , (1/3)), 'gamma': np.power(((n_arms * np.log(n_arms)) / T) , (1/3))}
    }
}

sim_results, arm_selections = mc_simulation(algorithms, sanity_env_params, T, epochs)

In [ ]:
plot_cumulative_rewards(sim_results, title='Task A.2: Cumulative Rewards', filename='cum_reward_ete_a2.pdf')
plot_arm_selections(arm_selections, max_time=None, num_bins=200, title='Task A.2: General Histogram', filename='selections_ete_a2.pdf')

#### A.2 Instance where MW beats UCB

In [ ]:
T = 50000
epochs = 50

n_arms = 2

sanity_env_params = {
    'p_L': [0.1, 0.1],
    'p_H': [0.9, 0.9],
    'T_L': [1, 1],
    'T_H': [1, 1],
    'initial_states': ['L', 'H']
}

algorithms = {
    'Upper Confidence Bound': {
        'class': UCB,
        'params': {'n_arms': n_arms}
    },
    'Multiplicative Weights in Bandit mode': {
        'class': MWU_bandit,
        'params': {'n_arms': n_arms, 'h': np.power(((n_arms * np.log(n_arms)) / T) , (1/3)), 'gamma': np.power(((n_arms * np.log(n_arms)) / T) , (1/3))}
    }
}

sim_results, arm_selections = mc_simulation(algorithms, sanity_env_params, T, epochs)

In [ ]:
plot_cumulative_rewards(sim_results, title='Task A.2: Cumulative Rewards', filename='cum_reward_ucb_a2.pdf')
plot_arm_selections(arm_selections, max_time=None, num_bins=200, title='Task A.2: General Histogram', filename='selections_ucb_a2.pdf')

# Part B

#### Environment implementation

###### Environment class

In [ ]:
class PriceWarEnvironment:
    def __init__(self, K, beta, seed):
        self.K = K
        self.beta = beta

        np.random.seed(seed)
        self.learner_prices = np.sort(np.random.uniform(0, 5, self.K))

    def customer_choice_prob(self, p_t, q_t):
        prob = 1.0 / (1.0 + np.exp(self.beta * (p_t - q_t)))
        return prob

    def buy(self, p_t, q_t):
        prob_learner = self.customer_choice_prob(p_t, q_t)

        learner_chosen = np.random.binomial(1, prob_learner)

        return (p_t, 0.0) if learner_chosen else (0.0, q_t)

###### Environment simulation

In [ ]:
def run_price_war_sim(beta, seed, T=5000, epochs=50, K=10):
    results = {
        'UCB': {'learner_reward': np.zeros((epochs, T)), 'comp_reward': np.zeros((epochs, T)), 
                'learner_prices': np.zeros((epochs, T)), 'comp_prices': np.zeros((epochs, T))},
        'MW':  {'learner_reward': np.zeros((epochs, T)), 'comp_reward': np.zeros((epochs, T)), 
                'learner_prices': np.zeros((epochs, T)), 'comp_prices': np.zeros((epochs, T))}
    }

    mwu_param = np.power(((K * np.log(K)) / T) , (1/3))

    env = PriceWarEnvironment(K=K, beta=beta, seed=seed)
    competitor = Competitor(env=env)

    for epoch in range(epochs):
        
        ucb_agent = UCB(K)
        np.random.seed(seed + epoch)
        for t in range(T):
            chosen_price = ucb_agent.select_arm()
            p_t = env.learner_prices[chosen_price]

            q_t = competitor.best_response_ucb(p_t)
            reward_ucb, reward_comp = env.buy(p_t, q_t)

            ucb_agent.update(chosen_price, reward_ucb)

            results['UCB']['learner_reward'][epoch, t] = reward_ucb
            results['UCB']['comp_reward'][epoch, t] = reward_comp
            results['UCB']['learner_prices'][epoch, t] = p_t
            results['UCB']['comp_prices'][epoch, t] = q_t

        mw_agent = MWU_bandit(K, mwu_param, mwu_param)
        np.random.seed(seed + epoch)
        for t in range(T):
            chosen_price = mw_agent.select_arm()
            p_t = env.learner_prices[chosen_price]

            q_t = competitor.best_response_mw(mw_agent.get_probabilities())
            reward_mw, reward_comp = env.buy(p_t, q_t)

            ucb_agent.update(chosen_price, reward_mw)

            results['MW']['learner_reward'][epoch, t] = reward_mw
            results['MW']['comp_reward'][epoch, t] = reward_comp
            results['MW']['learner_prices'][epoch, t] = p_t
            results['MW']['comp_prices'][epoch, t] = q_t

    return results

###### Figure plotting

In [ ]:
def report_price_war_results(results_dict, beta, smoothing_window=50, filename=None):
    print(f"\n{'='*50}")
    print(f"RESULTS FOR PRICE SENSITIVITY: β = {beta}")
    print(f"{'='*50}")
    
    for algo in ['UCB', 'MW']:
        learner_totals = np.sum(results_dict[algo]['learner_reward'], axis=1)
        comp_totals = np.sum(results_dict[algo]['comp_reward'], axis=1)
        
        l_mean, l_std = np.mean(learner_totals), np.std(learner_totals)
        c_mean, c_std = np.mean(comp_totals), np.std(comp_totals)
        
        print(f"[{algo} vs Competitor]")
        print(f"  Learner Final Reward    : {l_mean:.2f} ± {l_std:.2f}")
        print(f"  Competitor Final Reward : {c_mean:.2f} ± {c_std:.2f}\n")
        
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    
    def smooth(y, window):
        box = np.ones(window) / window
        return np.convolve(y, box, mode='valid')

    for algo, color in zip(['UCB', 'MW'], ['blue', 'green']):
        learner_cum = np.mean(np.cumsum(results_dict[algo]['learner_reward'], axis=1), axis=0)
        comp_cum = np.mean(np.cumsum(results_dict[algo]['comp_reward'], axis=1), axis=0)
        
        axes[0].plot(learner_cum, label=f'{algo} Learner', color=color)
        axes[0].plot(comp_cum, label=f'Competitor vs {algo}', color=color, linestyle=':')

    axes[0].set_title(f"Cumulative Rewards Over Time (β={beta})")
    axes[0].set_xlabel("Time Steps (t)")
    axes[0].set_ylabel("MC Mean Cumulative Reward")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    for algo, color in zip(['UCB', 'MW'], ['blue', 'green']):
        learner_prices_mean = np.mean(results_dict[algo]['learner_prices'], axis=0)
        comp_prices_mean = np.mean(results_dict[algo]['comp_prices'], axis=0)
        
        axes[1].plot(smooth(learner_prices_mean, smoothing_window), 
                     label=f'{algo} Learner Price', color=color)
        axes[1].plot(smooth(comp_prices_mean, smoothing_window), 
                     label=f'Competitor vs {algo} Price', color=color, linestyle=':')

    axes[1].set_title(f"Price Evolution (Smoothed Window={smoothing_window})")
    axes[1].set_xlabel("Time Steps (t)")
    axes[1].set_ylabel("MC Mean Price")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    
    if filename:
        plt.savefig(filename, format='pdf',bbox_inches='tight')
        
    plt.show()

#### Competitor Algorithm

In [ ]:
class Competitor:
    def __init__(self, env, q_max=10, vector_len=20000):
        self.env = env
        self.q_vector = np.linspace(0, q_max, vector_len)

    def best_response_ucb(self, p_t):
        prob_learner = self.env.customer_choice_prob(p_t, self.q_vector)
        prob_comp = 1 - prob_learner
        expected_rewards = self.q_vector * prob_comp

        best_idx = np.argmax(expected_rewards)
        return self.q_vector[best_idx]

    def best_response_mw(self, prob_vector):
        P = self.env.learner_prices.reshape(-1, 1)
        Q = self.q_vector.reshape(1, -1)

        prob_learner_matrix = 1.0 / (1.0 + np.exp(self.env.beta * (P - Q)))
        prob_comp_matrix = 1.0 - prob_learner_matrix

        reward_matrix = Q * prob_comp_matrix

        expected_rewards = np.dot(prob_vector, reward_matrix)

        best_idx = np.argmax(expected_rewards)
        return self.q_vector[best_idx]

#### Price Wars!!

In [ ]:
seed = int(str(STUDENT_ID)[-2:])

results_low = run_price_war_sim(beta=0.5, T=5000, epochs=30, seed=seed)
# 2. Medium Sensitivity Regime
results_med = run_price_war_sim(beta=2.0, T=5000, epochs=30, seed=seed)
# 3. High Sensitivity Regime (MW should dominate UCB)
results_high = run_price_war_sim(beta=10.0, T=5000, epochs=30, seed=seed)

In [ ]:
report_price_war_results(results_low, beta=0.5, filename='price_low.pdf')
report_price_war_results(results_med, beta=2.0, filename='price_med.pdf')
report_price_war_results(results_high, beta=10.0, filename='price_high.pdf')